# QwenFree Kaggle server

1. In Notebook options, select **Accelerator → GPU T4 x2** and turn **Internet** on.
2. Add and attach the recommended `NGROK_AUTHTOKEN` Secret for the primary
   tunnel. `HF_TOKEN` and `QWENFREE_API_KEY` are optional.
3. Review the repository and context values in the next cell, then run all cells.
4. Copy the printed `qwenfree connect` command into Terminal on your Mac.

The notebook prefers ngrok when the secret provides a URL and falls back to
Cloudflare Quick Tunnel. Cloudflare Quick Tunnels do not support Server-Sent
Events, so QwenFree uses a completed response for `trycloudflare.com` URLs.

The final cell runs the authenticated server until you interrupt it. It reports
health every five minutes and does not try to bypass Kaggle session or quota limits.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

def optional_secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for name in ('HF_TOKEN', 'QWENFREE_API_KEY', 'NGROK_AUTHTOKEN'):
    value = optional_secret(name)
    if value:
        os.environ[name] = value

# Change this value to choose another verified Hugging Face GGUF repository.
os.environ.setdefault('MODEL_REPO', 'JonathanColetti/Qwen3.8-27B-Uncensored-GGUF')
os.environ.setdefault('CTX_SIZE', '8192')
os.environ.setdefault('MAX_MODEL_GIB', '24')
print('Secret values were loaded without displaying them.')
print('Selected model repository:', os.environ['MODEL_REPO'])


In [ ]:
from pathlib import Path
import base64

server_dir = Path('/kaggle/working/qwenfree_server')
server_dir.mkdir(parents=True, exist_ok=True)
embedded_files = {
    "bootstrap.sh": "IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLUVldW8gcGlwZWZhaWwKClNDUklQVF9ESVI9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkKSIKV09SS19ESVI9IiR7V09SS19ESVI6LS9rYWdnbGUvd29ya2luZ30iCkxMQU1BX0NQUF9ESVI9IiR7TExBTUFfQ1BQX0RJUjotJHtXT1JLX0RJUn0vbGxhbWEuY3BwfSIKTU9ERUxfRElSPSIke01PREVMX0RJUjotL2thZ2dsZS90ZW1wL21vZGVsfSIKQklOX0RJUj0iJHtXT1JLX0RJUn0vcXdlbmZyZWUtYmluIgpMT0dfRElSPSIke1dPUktfRElSfS9xd2VuZnJlZS1sb2dzIgpTRVJWRVJfQklOPSIke0xMQU1BX0NQUF9ESVJ9L2J1aWxkL2Jpbi9sbGFtYS1zZXJ2ZXIiCkNUWF9TSVpFPSIke0NUWF9TSVpFOi04MTkyfSIKTUFYX01PREVMX0dJQj0iJHtNQVhfTU9ERUxfR0lCOi0yNH0iCk1PREVMX1JFUE89IiR7TU9ERUxfUkVQTzotSm9uYXRoYW5Db2xldHRpL1F3ZW4zLjgtMjdCLVVuY2Vuc29yZWQtR0dVRn0iCmV4cG9ydCBNT0RFTF9SRVBPIE1BWF9NT0RFTF9HSUIKClNFUlZFUl9QSUQ9IiIKQ0xPVURGTEFSRURfUElEPSIiCk5HUk9LX1BJRD0iIgoKY2xlYW51cCgpIHsKICBsb2NhbCBleGl0X2NvZGU9JD8KICB0cmFwIC0gRVhJVCBJTlQgVEVSTQogIGVjaG8gIlN0b3BwaW5nIFF3ZW5GcmVlIHNlcnZlciBhbmQgdHVubmVsIHByb2Nlc3Nlcy4uLiIKICBmb3IgcHJvY2Vzc19pZCBpbiAiJHtDTE9VREZMQVJFRF9QSUR9IiAiJHtOR1JPS19QSUR9IiAiJHtTRVJWRVJfUElEfSI7IGRvCiAgICBpZiBbWyAtbiAiJHtwcm9jZXNzX2lkfSIgXV0gJiYga2lsbCAtMCAiJHtwcm9jZXNzX2lkfSIgMj4vZGV2L251bGw7IHRoZW4KICAgICAga2lsbCAiJHtwcm9jZXNzX2lkfSIgMj4vZGV2L251bGwgfHwgdHJ1ZQogICAgZmkKICBkb25lCiAgZm9yIHByb2Nlc3NfaWQgaW4gIiR7Q0xPVURGTEFSRURfUElEfSIgIiR7TkdST0tfUElEfSIgIiR7U0VSVkVSX1BJRH0iOyBkbwogICAgaWYgW1sgLW4gIiR7cHJvY2Vzc19pZH0iIF1dOyB0aGVuCiAgICAgIHdhaXQgIiR7cHJvY2Vzc19pZH0iIDI+L2Rldi9udWxsIHx8IHRydWUKICAgIGZpCiAgZG9uZQogIGV4aXQgIiR7ZXhpdF9jb2RlfSIKfQp0cmFwIGNsZWFudXAgRVhJVAp0cmFwICdleGl0IDEzMCcgSU5UCnRyYXAgJ2V4aXQgMTQzJyBURVJNCgpmb3IgZGVwZW5kZW5jeSBpbiBnaXQgY21ha2UgY3VybCBweXRob24zOyBkbwogIGlmICEgY29tbWFuZCAtdiAiJHtkZXBlbmRlbmN5fSIgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBlY2hvICJSZXF1aXJlZCBjb21tYW5kIG5vdCBmb3VuZDogJHtkZXBlbmRlbmN5fS4gVXNlIGEgS2FnZ2xlIEdQVSBub3RlYm9vay4iID4mMgogICAgZXhpdCAxCiAgZmkKZG9uZQoKaWYgISBjb21tYW5kIC12IG52aWRpYS1zbWkgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgZWNobyAibnZpZGlhLXNtaSBpcyBtaXNzaW5nOyBlbmFibGUgQWNjZWxlcmF0b3IgLT4gR1BVIFQ0IHgyIGluIEthZ2dsZSBzZXR0aW5ncy4iID4mMgogIGV4aXQgMQpmaQpHUFVfQ09VTlQ9IiQobnZpZGlhLXNtaSAtTCB8IGdyZXAgLWMgJ15HUFUgJyB8fCB0cnVlKSIKaWYgW1sgIiR7R1BVX0NPVU5UfSIgIT0gIjIiIF1dOyB0aGVuCiAgZWNobyAiRXhwZWN0ZWQgZXhhY3RseSAyIEdQVXMsIGZvdW5kICR7R1BVX0NPVU5UfS4gU2VsZWN0IEdQVSBUNCB4MiBhbmQgcmVydW4uIiA+JjIKICBudmlkaWEtc21pIC1MIHx8IHRydWUKICBleGl0IDEKZmkKZWNobyAiRGV0ZWN0ZWQgdHdvIEdQVXM6IgpudmlkaWEtc21pIC0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsIC0tZm9ybWF0PWNzdixub2hlYWRlcgoKaWYgW1sgISAiJHtDVFhfU0laRX0iID1+IF5bMC05XSskIF1dIHx8ICgoIENUWF9TSVpFIDwgMTAyNCB8fCBDVFhfU0laRSA+IDMyNzY4ICkpOyB0aGVuCiAgZWNobyAiQ1RYX1NJWkUgbXVzdCBiZSBhbiBpbnRlZ2VyIGZyb20gMTAyNCB0byAzMjc2ODsgbmF0aXZlIDI2MmsgY29udGV4dCBpcyBub3QgdXNlZC4iID4mMgogIGV4aXQgMQpmaQoKbWtkaXIgLXAgIiR7V09SS19ESVJ9IiAiJHtCSU5fRElSfSIgIiR7TE9HX0RJUn0iICIke01PREVMX0RJUn0iCmlmICEgcHl0aG9uMyAtYyAnaW1wb3J0IGh1Z2dpbmdmYWNlX2h1YicgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgZWNobyAiSW5zdGFsbGluZyB0aGUgc21hbGwgc2VydmVyLXNpZGUgSHVnZ2luZyBGYWNlIGNsaWVudC4uLiIKICBweXRob24zIC1tIHBpcCBpbnN0YWxsIC0tcXVpZXQgaHVnZ2luZ2ZhY2VfaHViCmZpCgppZiBbWyAtZCAiJHtMTEFNQV9DUFBfRElSfS8uZ2l0IiBdXTsgdGhlbgogIGVjaG8gIlVwZGF0aW5nIGxsYW1hLmNwcCBmcm9tIGl0cyBjb25maWd1cmVkIGRlZmF1bHQgYnJhbmNoLi4uIgogIGdpdCAtQyAiJHtMTEFNQV9DUFBfRElSfSIgcHVsbCAtLWZmLW9ubHkKZWxzZQogIGVjaG8gIkNsb25pbmcgdGhlIGxhdGVzdCBkZWZhdWx0IGJyYW5jaCBvZiBnZ21sLW9yZy9sbGFtYS5jcHAuLi4iCiAgZ2l0IGNsb25lIC0tZGVwdGggMSBodHRwczovL2dpdGh1Yi5jb20vZ2dtbC1vcmcvbGxhbWEuY3BwLmdpdCAiJHtMTEFNQV9DUFBfRElSfSIKZmkKCmVjaG8gIkJ1aWxkaW5nIGxsYW1hLXNlcnZlciB3aXRoIENVREEgZm9yIE5WSURJQSBUNCAoY29tcHV0ZSBjYXBhYmlsaXR5IDcuNSkuLi4iCmNtYWtlIC1TICIke0xMQU1BX0NQUF9ESVJ9IiAtQiAiJHtMTEFNQV9DUFBfRElSfS9idWlsZCIgXAogIC1ER0dNTF9DVURBPU9OIFwKICAtRENNQUtFX0NVREFfQVJDSElURUNUVVJFUz03NSBcCiAgLURDTUFLRV9CVUlMRF9UWVBFPVJlbGVhc2UKY21ha2UgLS1idWlsZCAiJHtMTEFNQV9DUFBfRElSfS9idWlsZCIgLS1jb25maWcgUmVsZWFzZSBcCiAgLS10YXJnZXQgbGxhbWEtc2VydmVyIC1qIDQKCmlmIFtbICEgLXggIiR7U0VSVkVSX0JJTn0iIF1dOyB0aGVuCiAgZWNobyAiQnVpbGQgY29tcGxldGVkLCBidXQgbGxhbWEtc2VydmVyIHdhcyBub3QgZm91bmQgYXQgJHtTRVJWRVJfQklOfS4iID4mMgogIGV4aXQgMQpmaQoKU0VMRUNUSU9OX0ZJTEU9IiR7V09SS19ESVJ9L3F3ZW5mcmVlLW1vZGVsLXNlbGVjdGlvbi5qc29uIgplY2hvICJDaGVja2luZyB0aGUgc2VsZWN0ZWQgSHVnZ2luZyBGYWNlIHJlcG9zaXRvcnkgYW5kIEdHVUYgc2l6ZXMuLi4iCnB5dGhvbjMgIiR7U0NSSVBUX0RJUn0vZmluZF9tb2RlbC5weSIgLS1qc29uIC0tbWF4LW1vZGVsLWdpYiAiJHtNQVhfTU9ERUxfR0lCfSIgXAogID4gIiR7U0VMRUNUSU9OX0ZJTEV9IgoKcHl0aG9uMyAtICIke1NFTEVDVElPTl9GSUxFfSIgIiR7TU9ERUxfRElSfSIgPDwnUFknCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBzbmFwc2hvdF9kb3dubG9hZAoKc2VsZWN0aW9uX3BhdGgsIG1vZGVsX2RpciA9IHN5cy5hcmd2WzE6XQp3aXRoIG9wZW4oc2VsZWN0aW9uX3BhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIHNvdXJjZToKICAgIHNlbGVjdGlvbiA9IGpzb24ubG9hZChzb3VyY2UpCnByaW50KAogICAgIkRvd25sb2FkaW5nIHt9IHt9ICh7Oi4yZn0gR2lCIG9mIHdlaWdodHMpIGZyb20ge30iLmZvcm1hdCgKICAgICAgICBzZWxlY3Rpb25bInF1YW50Il0sCiAgICAgICAgc2VsZWN0aW9uWyJwcmltYXJ5X2ZpbGUiXSwKICAgICAgICBzZWxlY3Rpb25bInRvdGFsX3NpemVfYnl0ZXMiXSAvIDEwMjQqKjMsCiAgICAgICAgc2VsZWN0aW9uWyJyZXBvX2lkIl0sCiAgICApCikKc25hcHNob3RfZG93bmxvYWQoCiAgICByZXBvX2lkPXNlbGVjdGlvblsicmVwb19pZCJdLAogICAgbG9jYWxfZGlyPW1vZGVsX2RpciwKICAgIGFsbG93X3BhdHRlcm5zPXNlbGVjdGlvblsiZmlsZXMiXSwKICAgIHRva2VuPW9zLmVudmlyb24uZ2V0KCJIRl9UT0tFTiIpIG9yIE5vbmUsCikKZm9yIGZpbGVuYW1lIGluIHNlbGVjdGlvblsiZmlsZXMiXToKICAgIGlmIG5vdCBvcy5wYXRoLmlzZmlsZShvcy5wYXRoLmpvaW4obW9kZWxfZGlyLCBmaWxlbmFtZSkpOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIk1vZGVsIGRvd25sb2FkIGlzIGluY29tcGxldGU6ICIgKyBmaWxlbmFtZSkKUFkKClBSSU1BUllfRklMRT0iJChweXRob24zIC1jICdpbXBvcnQganNvbixzeXM7IHByaW50KGpzb24ubG9hZChvcGVuKHN5cy5hcmd2WzFdKSlbInByaW1hcnlfZmlsZSJdKScgIiR7U0VMRUNUSU9OX0ZJTEV9IikiCk1PREVMX1BBVEg9IiR7TU9ERUxfRElSfS8ke1BSSU1BUllfRklMRX0iCmlmIFtbICEgLXMgIiR7TU9ERUxfUEFUSH0iIF1dOyB0aGVuCiAgZWNobyAiU2VsZWN0ZWQgbW9kZWwgZmlsZSBpcyBtaXNzaW5nIG9yIGVtcHR5OiAke01PREVMX1BBVEh9IiA+JjIKICBleGl0IDEKZmkKClNFUlZFUl9IRUxQPSIkKCIke1NFUlZFUl9CSU59IiAtLWhlbHAgMj4mMSB8fCB0cnVlKSIKcmVxdWlyZV9zZXJ2ZXJfb3B0aW9uKCkgewogIGxvY2FsIG9wdGlvbj0iJDEiCiAgaWYgISBncmVwIC1GcSAtLSAiJHtvcHRpb259IiA8PDwiJHtTRVJWRVJfSEVMUH0iOyB0aGVuCiAgICBlY2hvICJUaGUgYnVpbHQgbGxhbWEtc2VydmVyIGRvZXMgbm90IHN1cHBvcnQgcmVxdWlyZWQgb3B0aW9uICR7b3B0aW9ufS4iID4mMgogICAgZWNobyAiUmVidWlsZCBmcm9tIHRoZSBsYXRlc3QgbGxhbWEuY3BwIGRlZmF1bHQgYnJhbmNoIGFuZCB0cnkgYWdhaW4uIiA+JjIKICAgIGV4aXQgMQogIGZpCn0KZm9yIG9wdGlvbiBpbiAtLXRlbnNvci1zcGxpdCAtLWZsYXNoLWF0dG4gLS1jdHgtc2l6ZSAtLWhvc3QgLS1wb3J0IC0tYXBpLWtleSAtLWppbmphOyBkbwogIHJlcXVpcmVfc2VydmVyX29wdGlvbiAiJHtvcHRpb259Igpkb25lCgpTRVJWRVJfQVJHUz0oCiAgLW0gIiR7TU9ERUxfUEFUSH0iCiAgLW5nbCA5OQogIC0tdGVuc29yLXNwbGl0IDEsMQogIC1jICIke0NUWF9TSVpFfSIKICAtLWhvc3QgMTI3LjAuMC4xCiAgLS1wb3J0IDgwODAKICAtLWppbmphCikKaWYgZ3JlcCAtRnEgLS0gJy0tZmxhc2gtYXR0biBbb258b2ZmfGF1dG9dJyA8PDwiJHtTRVJWRVJfSEVMUH0iIHx8IFwKICBncmVwIC1GcSAtLSAnLS1mbGFzaC1hdHRuIDxvbnxvZmZ8YXV0bz4nIDw8PCIke1NFUlZFUl9IRUxQfSI7IHRoZW4KICBTRVJWRVJfQVJHUys9KC0tZmxhc2gtYXR0biBvbikKZWxzZQogIFNFUlZFUl9BUkdTKz0oLS1mbGFzaC1hdHRuKQpmaQppZiBncmVwIC1GcSAtLSAnLS1jYWNoZS10eXBlLWsnIDw8PCIke1NFUlZFUl9IRUxQfSIgJiYgXAogIGdyZXAgLUZxIC0tICctLWNhY2hlLXR5cGUtdicgPDw8IiR7U0VSVkVSX0hFTFB9IiAmJiBcCiAgZ3JlcCAtRnEgJ3E4XzAnIDw8PCIke1NFUlZFUl9IRUxQfSI7IHRoZW4KICBTRVJWRVJfQVJHUys9KC0tY2FjaGUtdHlwZS1rIHE4XzAgLS1jYWNoZS10eXBlLXYgcThfMCkKICBlY2hvICJVc2luZyBxOF8wIEtWIGNhY2hlIGJlY2F1c2UgdGhpcyBidWlsZCBhZHZlcnRpc2VzIGJvdGggY2FjaGUgZmxhZ3MuIgplbHNlCiAgZWNobyAiVGhpcyBsbGFtYS1zZXJ2ZXIgYnVpbGQgZG9lcyBub3QgYWR2ZXJ0aXNlIHE4XzAgS1YgY2FjaGU7IHVzaW5nIGl0cyBkZWZhdWx0IEtWIGNhY2hlLiIKZmkKaWYgZ3JlcCAtRnEgLS0gJy0tcGFyYWxsZWwnIDw8PCIke1NFUlZFUl9IRUxQfSI7IHRoZW4KICBTRVJWRVJfQVJHUys9KC0tcGFyYWxsZWwgMSkKZmkKCmlmIFtbIC1uICIke1FXRU5GUkVFX0FQSV9LRVk6LX0iIF1dOyB0aGVuCiAgQVBJX0tFWT0iJHtRV0VORlJFRV9BUElfS0VZfSIKZWxzZQogIEFQSV9LRVk9IiQocHl0aG9uMyAtYyAnaW1wb3J0IHNlY3JldHM7IHByaW50KHNlY3JldHMudG9rZW5fdXJsc2FmZSgzMikpJykiCmZpCgplY2hvICJTdGFydGluZyBsbGFtYS1zZXJ2ZXIgb24gbG9vcGJhY2suLi4iCiIke1NFUlZFUl9CSU59IiAiJHtTRVJWRVJfQVJHU1tAXX0iIC0tYXBpLWtleSAiJHtBUElfS0VZfSIgXAogID4gIiR7TE9HX0RJUn0vbGxhbWEtc2VydmVyLmxvZyIgMj4mMSAmClNFUlZFUl9QSUQ9JCEKClNFUlZFUl9SRUFEWT0iZmFsc2UiCmZvciBfIGluICQoc2VxIDEgMTgwKTsgZG8KICBpZiAhIGtpbGwgLTAgIiR7U0VSVkVSX1BJRH0iIDI+L2Rldi9udWxsOyB0aGVuCiAgICBicmVhawogIGZpCiAgaWYgY3VybCAtLXNpbGVudCAtLXNob3ctZXJyb3IgLS1mYWlsIC0tbWF4LXRpbWUgNSBcCiAgICBodHRwOi8vMTI3LjAuMC4xOjgwODAvaGVhbHRoID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgU0VSVkVSX1JFQURZPSJ0cnVlIgogICAgYnJlYWsKICBmaQogIHNsZWVwIDUKZG9uZQppZiBbWyAiJHtTRVJWRVJfUkVBRFl9IiAhPSAidHJ1ZSIgXV07IHRoZW4KICBpZiBncmVwIC1FcWkgJ3Vua25vd24gKG1vZGVsICk/YXJjaGl0ZWN0dXJlJyAiJHtMT0dfRElSfS9sbGFtYS1zZXJ2ZXIubG9nIjsgdGhlbgogICAgZWNobyAiVW5rbm93biBtb2RlbCBhcmNoaXRlY3R1cmU6IHJlYnVpbGQgbGxhbWEuY3BwIGZyb20gaXRzIGxhdGVzdCBtYXN0ZXIvZGVmYXVsdCBicmFuY2guIiA+JjIKICBlbHNlCiAgICBlY2hvICJsbGFtYS1zZXJ2ZXIgZGlkIG5vdCBiZWNvbWUgaGVhbHRoeS4gUmVjZW50IHNlcnZlciBsb2cgZm9sbG93czoiID4mMgogIGZpCiAgdGFpbCAtbiA2MCAiJHtMT0dfRElSfS9sbGFtYS1zZXJ2ZXIubG9nIiA+JjIgfHwgdHJ1ZQogIGV4aXQgMQpmaQoKQ0xPVURGTEFSRURfQklOPSIke0JJTl9ESVJ9L2Nsb3VkZmxhcmVkIgppZiBbWyAhIC14ICIke0NMT1VERkxBUkVEX0JJTn0iIF1dOyB0aGVuCiAgZWNobyAiRG93bmxvYWRpbmcgY2xvdWRmbGFyZWQgbGludXgtYW1kNjQgZnJvbSBDbG91ZGZsYXJlJ3Mgb2ZmaWNpYWwgR2l0SHViIHJlbGVhc2UuLi4iCiAgY3VybCAtLWZhaWwgLS1sb2NhdGlvbiAtLXNpbGVudCAtLXNob3ctZXJyb3IgXAogICAgaHR0cHM6Ly9naXRodWIuY29tL2Nsb3VkZmxhcmUvY2xvdWRmbGFyZWQvcmVsZWFzZXMvbGF0ZXN0L2Rvd25sb2FkL2Nsb3VkZmxhcmVkLWxpbnV4LWFtZDY0IFwKICAgIC0tb3V0cHV0ICIke0NMT1VERkxBUkVEX0JJTn0iCiAgY2htb2QgNzAwICIke0NMT1VERkxBUkVEX0JJTn0iCmZpCiIke0NMT1VERkxBUkVEX0JJTn0iIC0tdmVyc2lvbgoiJHtDTE9VREZMQVJFRF9CSU59IiB0dW5uZWwgLS11cmwgaHR0cDovLzEyNy4wLjAuMTo4MDgwIFwKICA+ICIke0xPR19ESVJ9L2Nsb3VkZmxhcmVkLmxvZyIgMj4mMSAmCkNMT1VERkxBUkVEX1BJRD0kIQoKUFVCTElDX1VSTD0iIgpmb3IgXyBpbiAkKHNlcSAxIDMwKTsgZG8KICBQVUJMSUNfVVJMPSIkKGdyZXAgLUVvICdodHRwczovL1tbOmFsbnVtOl0tXStcLnRyeWNsb3VkZmxhcmVcLmNvbScgXAogICAgIiR7TE9HX0RJUn0vY2xvdWRmbGFyZWQubG9nIiB8IGhlYWQgLW4gMSB8fCB0cnVlKSIKICBpZiBbWyAtbiAiJHtQVUJMSUNfVVJMfSIgXV07IHRoZW4KICAgIGJyZWFrCiAgZmkKICBpZiAhIGtpbGwgLTAgIiR7Q0xPVURGTEFSRURfUElEfSIgMj4vZGV2L251bGw7IHRoZW4KICAgIGJyZWFrCiAgZmkKICBzbGVlcCAyCmRvbmUKCmlmIFtbIC1uICIke05HUk9LX0FVVEhUT0tFTjotfSIgXV07IHRoZW4KICBpZiBbWyAtbiAiJHtQVUJMSUNfVVJMfSIgXV07IHRoZW4KICAgIGVjaG8gIk5HUk9LX0FVVEhUT0tFTiBpcyBjb25maWd1cmVkOyB0cnlpbmcgbmdyb2sgZm9yIHRva2VuIHN0cmVhbWluZy4iCiAgICBlY2hvICJDbG91ZGZsYXJlIFF1aWNrIFR1bm5lbHMgZG8gbm90IHN1cHBvcnQgU2VydmVyLVNlbnQgRXZlbnRzLiIKICBlbHNlCiAgICBlY2hvICJDbG91ZGZsYXJlIHF1aWNrIHR1bm5lbCBkaWQgbm90IHByb3ZpZGUgYSBVUkw7IHRyeWluZyBvcHRpb25hbCBuZ3JvayBmYWxsYmFjay4uLiIKICBmaQogIE5HUk9LX0RJUj0iJHtCSU5fRElSfS9uZ3JvayIKICBta2RpciAtcCAiJHtOR1JPS19ESVJ9IgogIGlmIFtbICEgLXggIiR7TkdST0tfRElSfS9uZ3JvayIgXV07IHRoZW4KICAgIGN1cmwgLS1mYWlsIC0tbG9jYXRpb24gLS1zaWxlbnQgLS1zaG93LWVycm9yIFwKICAgICAgaHR0cHM6Ly9iaW4ubmdyb2suY29tL2MvYk55ajFtUVZZNGMvbmdyb2stdjMtc3RhYmxlLWxpbnV4LWFtZDY0LnRneiBcCiAgICAgIC0tb3V0cHV0ICIke05HUk9LX0RJUn0vbmdyb2sudGd6IgogICAgdGFyIC14emYgIiR7TkdST0tfRElSfS9uZ3Jvay50Z3oiIC1DICIke05HUk9LX0RJUn0iCiAgICBjaG1vZCA3MDAgIiR7TkdST0tfRElSfS9uZ3JvayIKICBmaQogIE5HUk9LX0NPTkZJRz0iJHtXT1JLX0RJUn0vcXdlbmZyZWUtbmdyb2sueW1sIgogICIke05HUk9LX0RJUn0vbmdyb2siIGNvbmZpZyBhZGQtYXV0aHRva2VuICIke05HUk9LX0FVVEhUT0tFTn0iIFwKICAgIC0tY29uZmlnICIke05HUk9LX0NPTkZJR30iID4vZGV2L251bGwKICBjaG1vZCA2MDAgIiR7TkdST0tfQ09ORklHfSIKICAiJHtOR1JPS19ESVJ9L25ncm9rIiBodHRwIDgwODAgLS1jb25maWcgIiR7TkdST0tfQ09ORklHfSIgXAogICAgLS1sb2cgc3Rkb3V0IC0tbG9nLWZvcm1hdCBqc29uID4gIiR7TE9HX0RJUn0vbmdyb2subG9nIiAyPiYxICYKICBOR1JPS19QSUQ9JCEKICBOR1JPS19VUkw9IiIKICBmb3IgXyBpbiAkKHNlcSAxIDMwKTsgZG8KICAgIE5HUk9LX1VSTD0iJChweXRob24zIC0gPDwnUFknCmltcG9ydCBqc29uCmltcG9ydCB1cmxsaWIuZXJyb3IKaW1wb3J0IHVybGxpYi5yZXF1ZXN0Cgp0cnk6CiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oImh0dHA6Ly8xMjcuMC4wLjE6NDA0MC9hcGkvdHVubmVscyIsIHRpbWVvdXQ9MSkgYXMgcmVzcG9uc2U6CiAgICAgICAgdHVubmVscyA9IGpzb24ubG9hZChyZXNwb25zZSkuZ2V0KCJ0dW5uZWxzIiwgW10pCiAgICBwcmludChuZXh0KChpdGVtWyJwdWJsaWNfdXJsIl0gZm9yIGl0ZW0gaW4gdHVubmVscyBpZiBpdGVtLmdldCgicHJvdG8iKSA9PSAiaHR0cHMiKSwgIiIpKQpleGNlcHQgKE9TRXJyb3IsIHVybGxpYi5lcnJvci5VUkxFcnJvciwganNvbi5KU09ORGVjb2RlRXJyb3IpOgogICAgcHJpbnQoIiIpClBZCiAgICApIgogICAgaWYgW1sgLW4gIiR7TkdST0tfVVJMfSIgXV07IHRoZW4KICAgICAgYnJlYWsKICAgIGZpCiAgICBzbGVlcCAyCiAgZG9uZQogIGlmIFtbIC1uICIke05HUk9LX1VSTH0iIF1dOyB0aGVuCiAgICBQVUJMSUNfVVJMPSIke05HUk9LX1VSTH0iCiAgICBpZiBbWyAtbiAiJHtDTE9VREZMQVJFRF9QSUR9IiBdXSAmJiBraWxsIC0wICIke0NMT1VERkxBUkVEX1BJRH0iIDI+L2Rldi9udWxsOyB0aGVuCiAgICAgIGtpbGwgIiR7Q0xPVURGTEFSRURfUElEfSIgMj4vZGV2L251bGwgfHwgdHJ1ZQogICAgICB3YWl0ICIke0NMT1VERkxBUkVEX1BJRH0iIDI+L2Rldi9udWxsIHx8IHRydWUKICAgICAgQ0xPVURGTEFSRURfUElEPSIiCiAgICBmaQogIGVsaWYgW1sgLXogIiR7UFVCTElDX1VSTH0iIF1dOyB0aGVuCiAgICBlY2hvICJuZ3JvayBkaWQgbm90IHByb3ZpZGUgYSBVUkw7IGNoZWNrICR7TE9HX0RJUn0vbmdyb2subG9nLiIgPiYyCiAgZmkKZmkKCmlmIFtbIC16ICIke1BVQkxJQ19VUkx9IiBdXTsgdGhlbgogIGVjaG8gIk5vIHB1YmxpYyB0dW5uZWwgVVJMIHdhcyBvYnRhaW5lZC4gUmVjZW50IHR1bm5lbCBsb2cgZm9sbG93czoiID4mMgogIHRhaWwgLW4gNDAgIiR7TE9HX0RJUn0vY2xvdWRmbGFyZWQubG9nIiA+JjIgfHwgdHJ1ZQogIGlmIFtbIC16ICIke05HUk9LX0FVVEhUT0tFTjotfSIgXV07IHRoZW4KICAgIGVjaG8gIlNldCB0aGUgb3B0aW9uYWwgTkdST0tfQVVUSFRPS0VOIEthZ2dsZSBTZWNyZXQgdG8gZW5hYmxlIG5ncm9rIGZhbGxiYWNrLiIgPiYyCiAgZmkKICBleGl0IDEKZmkKCmNhdCA8PEVPRgoKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09ClF3ZW5GcmVlIHNlcnZlciBpcyByZWFkeS4gS2VlcCB0aGlzIEthZ2dsZSBjZWxsIHJ1bm5pbmcuClFXRU5GUkVFX1VSTD0ke1BVQkxJQ19VUkx9ClFXRU5GUkVFX0tFWT0ke0FQSV9LRVl9CgpPbiB5b3VyIE1hYywgcnVuOgpxd2VuZnJlZSBjb25uZWN0ICR7UFVCTElDX1VSTH0gLS1rZXkgJHtBUElfS0VZfQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KRU9GCgp3aGlsZSBraWxsIC0wICIke1NFUlZFUl9QSUR9IiAyPi9kZXYvbnVsbDsgZG8KICBzbGVlcCAzMDAKICBpZiBjdXJsIC0tc2lsZW50IC0tc2hvdy1lcnJvciAtLWZhaWwgLS1tYXgtdGltZSAxMCBcCiAgICBodHRwOi8vMTI3LjAuMC4xOjgwODAvaGVhbHRoID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgZWNobyAiJChkYXRlIC11ICcrJVktJW0tJWQgJUg6JU06JVMgVVRDJykgUXdlbkZyZWUgc2VydmVyIGhlYWx0aDogT0siCiAgZWxzZQogICAgZWNobyAiJChkYXRlIC11ICcrJVktJW0tJWQgJUg6JU06JVMgVVRDJykgUXdlbkZyZWUgc2VydmVyIGhlYWx0aDogdW5hdmFpbGFibGUiCiAgZmkKZG9uZQoKZWNobyAibGxhbWEtc2VydmVyIHN0b3BwZWQ7IGNoZWNrICR7TE9HX0RJUn0vbGxhbWEtc2VydmVyLmxvZy4iCg==",
    "find_model.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTZWxlY3QgYSB2ZXJpZmllZCBRd2VuMy44LTI3QiB1bmNlbnNvcmVkIEdHVUYgZnJvbSBIdWdnaW5nIEZhY2UuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUHVyZVBvc2l4UGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpERUZBVUxUX1JFUE8gPSAiSm9uYXRoYW5Db2xldHRpL1F3ZW4zLjgtMjdCLVVuY2Vuc29yZWQtR0dVRiIKREVGQVVMVF9NQVhfTU9ERUxfR0lCID0gMjQuMApRVUFOVF9PUkRFUiA9ICgiUTRfS19NIiwgIklRNF9YUyIsICJRM19LX00iKQpHSUIgPSAxMDI0KiozCgoKY2xhc3MgTW9kZWxTZWxlY3Rpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gbm8gdmVyaWZpZWQgR0dVRiBmaXRzIHRoZSBjb25maWd1cmVkIG1lbW9yeSBidWRnZXQuIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgTW9kZWxTZWxlY3Rpb246CiAgICAiIiJUaGUgZXhhY3QgZmlsZXMgYW5kIG1vZGVsIG1ldGFkYXRhIG5lZWRlZCBieSB0aGUgYm9vdHN0cmFwIHNjcmlwdC4iIiIKCiAgICByZXBvX2lkOiBzdHIKICAgIHF1YW50OiBzdHIKICAgIGZpbGVzOiBsaXN0W3N0cl0KICAgIHByaW1hcnlfZmlsZTogc3RyCiAgICB0b3RhbF9zaXplX2J5dGVzOiBpbnQKICAgIG1heF9tb2RlbF9naWI6IGZsb2F0CgoKZGVmIF9nZXQoaXRlbTogQW55LCBrZXk6IHN0ciwgZGVmYXVsdDogQW55ID0gTm9uZSkgLT4gQW55OgogICAgIiIiUmVhZCBhIGZpZWxkIGZyb20gZWl0aGVyIEhGIG1ldGFkYXRhIG9iamVjdHMgb3IgdGVzdCBkaWN0aW9uYXJpZXMuIiIiCiAgICBpZiBpc2luc3RhbmNlKGl0ZW0sIGRpY3QpOgogICAgICAgIHJldHVybiBpdGVtLmdldChrZXksIGRlZmF1bHQpCiAgICByZXR1cm4gZ2V0YXR0cihpdGVtLCBrZXksIGRlZmF1bHQpCgoKZGVmIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBzdHIgfCBOb25lOgogICAgIiIiUmV0dXJuIGEgc3VwcG9ydGVkIHF1YW50IG5hbWUgd2hlbiBwcmVzZW50IGluIGEgR0dVRiBmaWxlbmFtZS4iIiIKICAgIGlmIG5vdCBmaWxlbmFtZS5sb3dlcigpLmVuZHN3aXRoKCIuZ2d1ZiIpOgogICAgICAgIHJldHVybiBOb25lCiAgICBsb3dlcmVkID0gZmlsZW5hbWUubG93ZXIoKQogICAgcGF0dGVybnMgPSB7CiAgICAgICAgIlE0X0tfTSI6IHIiKD88IVthLXowLTldKXE0W18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgICAgICAiSVE0X1hTIjogciIoPzwhW2EtejAtOV0paXE0W18tXT94cyg/IVthLXowLTldKSIsCiAgICAgICAgIlEzX0tfTSI6IHIiKD88IVthLXowLTldKXEzW18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgIH0KICAgIGZvciBxdWFudCwgcGF0dGVybiBpbiBwYXR0ZXJucy5pdGVtcygpOgogICAgICAgIGlmIHJlLnNlYXJjaChwYXR0ZXJuLCBsb3dlcmVkKToKICAgICAgICAgICAgcmV0dXJuIHF1YW50CiAgICByZXR1cm4gTm9uZQoKCmRlZiBfc2FmZV9yZXBvX2ZpbGVuYW1lKGZpbGVuYW1lOiBzdHIpIC0+IGJvb2w6CiAgICAiIiJSZWplY3QgYWJzb2x1dGUgcGF0aHMgYW5kIHRyYXZlcnNhbCBpbiB1bnRydXN0ZWQgSHViIGZpbGUgbWV0YWRhdGEuIiIiCiAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChmaWxlbmFtZSkKICAgIHJldHVybiBib29sKGZpbGVuYW1lKSBhbmQgbm90IHBhdGguaXNfYWJzb2x1dGUoKSBhbmQgIlxcIiBub3QgaW4gZmlsZW5hbWUgYW5kIGFsbCgKICAgICAgICBwYXJ0IG5vdCBpbiB7IiIsICIuIiwgIi4uIn0gZm9yIHBhcnQgaW4gcGF0aC5wYXJ0cwogICAgKQoKCmRlZiBfZ3JvdXBzX2Zvcl9xdWFudChzaWJsaW5nczogbGlzdFtBbnldLCBxdWFudDogc3RyKSAtPiBsaXN0W3R1cGxlW2xpc3Rbc3RyXSwgaW50XV06CiAgICAiIiJHcm91cCBhIHF1YW50J3MgZmlsZXMgaW50byBzaW5nbGUgR0dVRnMgb3IgY29tcGxldGUgc2hhcmQgc2V0cy4iIiIKICAgIGdyb3VwczogZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3Igc2libGluZyBpbiBzaWJsaW5nczoKICAgICAgICBmaWxlbmFtZSA9IHN0cihfZ2V0KHNpYmxpbmcsICJyZmlsZW5hbWUiLCAiIikpCiAgICAgICAgc2l6ZSA9IF9nZXQoc2libGluZywgInNpemUiKQogICAgICAgIGlmICgKICAgICAgICAgICAgbm90IF9zYWZlX3JlcG9fZmlsZW5hbWUoZmlsZW5hbWUpCiAgICAgICAgICAgIG9yIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZSkgIT0gcXVhbnQKICAgICAgICAgICAgb3Igbm90IGlzaW5zdGFuY2Uoc2l6ZSwgaW50KQogICAgICAgICAgICBvciBzaXplIDw9IDAKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNwbGl0ID0gcmUubWF0Y2goCiAgICAgICAgICAgIHIiXig/UDxzdGVtPi4qKS0oP1A8aW5kZXg+XGR7NX0pLW9mLSg/UDxjb3VudD5cZHs1fSlcLmdndWYkIiwKICAgICAgICAgICAgZmlsZW5hbWUsCiAgICAgICAgICAgIGZsYWdzPXJlLklHTk9SRUNBU0UsCiAgICAgICAgKQogICAgICAgIGlmIHNwbGl0OgogICAgICAgICAgICBzdGVtID0gc3BsaXQuZ3JvdXAoInN0ZW0iKS5sb3dlcigpCiAgICAgICAgICAgIGdyb3VwID0gZ3JvdXBzLnNldGRlZmF1bHQoCiAgICAgICAgICAgICAgICBzdGVtLAogICAgICAgICAgICAgICAgeyJmaWxlcyI6IFtdLCAiY291bnQiOiBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpLCAiaW5kaWNlcyI6IFtdfSwKICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBncm91cFsiY291bnQiXSAhPSBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZ3JvdXBbImZpbGVzIl0uYXBwZW5kKChpbnQoc3BsaXQuZ3JvdXAoImluZGV4IikpLCBmaWxlbmFtZSwgc2l6ZSkpCiAgICAgICAgICAgIGdyb3VwWyJpbmRpY2VzIl0uYXBwZW5kKGludChzcGxpdC5ncm91cCgiaW5kZXgiKSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZ3JvdXBzW2ZpbGVuYW1lLmxvd2VyKCldID0gewogICAgICAgICAgICAgICAgImZpbGVzIjogWygxLCBmaWxlbmFtZSwgc2l6ZSldLAogICAgICAgICAgICAgICAgImNvdW50IjogMSwKICAgICAgICAgICAgICAgICJpbmRpY2VzIjogWzFdLAogICAgICAgICAgICB9CgogICAgdmFsaWQ6IGxpc3RbdHVwbGVbbGlzdFtzdHJdLCBpbnRdXSA9IFtdCiAgICBmb3IgZ3JvdXAgaW4gZ3JvdXBzLnZhbHVlcygpOgogICAgICAgIGV4cGVjdGVkID0gZ3JvdXBbImNvdW50Il0KICAgICAgICBmaWxlc193aXRoX3NpemUgPSBncm91cFsiZmlsZXMiXQogICAgICAgIGluZGljZXMgPSBncm91cFsiaW5kaWNlcyJdCiAgICAgICAgaWYgbGVuKGZpbGVzX3dpdGhfc2l6ZSkgIT0gZXhwZWN0ZWQgb3Igc29ydGVkKGluZGljZXMpICE9IGxpc3QoCiAgICAgICAgICAgIHJhbmdlKDEsIGV4cGVjdGVkICsgMSkKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZpbGVzX3dpdGhfc2l6ZS5zb3J0KGtleT1sYW1iZGEgaXRlbTogaXRlbVswXSkKICAgICAgICBmaWxlbmFtZXMgPSBbaXRlbVsxXSBmb3IgaXRlbSBpbiBmaWxlc193aXRoX3NpemVdCiAgICAgICAgc2l6ZSA9IHN1bShpdGVtWzJdIGZvciBpdGVtIGluIGZpbGVzX3dpdGhfc2l6ZSkKICAgICAgICB2YWxpZC5hcHBlbmQoKGZpbGVuYW1lcywgc2l6ZSkpCiAgICByZXR1cm4gdmFsaWQKCgpkZWYgc2VsZWN0X3F1YW50KAogICAgcmVwb19pZDogc3RyLAogICAgc2libGluZ3M6IGxpc3RbQW55XSwKICAgICosCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlBpY2sgdGhlIHByZWZlcnJlZCBjb21wbGV0ZSBxdWFudCB0aGF0IGZpdHMgdW5kZXIgdGhlIHdlaWdodCBidWRnZXQuIiIiCiAgICBpZiBtYXhfbW9kZWxfZ2liIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X21vZGVsX2dpYiBtdXN0IGJlIHBvc2l0aXZlLiIpCiAgICBtYXhfYnl0ZXMgPSBpbnQobWF4X21vZGVsX2dpYiAqIEdJQikKICAgIGZvciBxdWFudCBpbiBRVUFOVF9PUkRFUjoKICAgICAgICBjYW5kaWRhdGVzID0gWwogICAgICAgICAgICBncm91cAogICAgICAgICAgICBmb3IgZ3JvdXAgaW4gX2dyb3Vwc19mb3JfcXVhbnQoc2libGluZ3MsIHF1YW50KQogICAgICAgICAgICBpZiBncm91cFsxXSA8PSBtYXhfYnl0ZXMKICAgICAgICBdCiAgICAgICAgaWYgY2FuZGlkYXRlczoKICAgICAgICAgICAgY2FuZGlkYXRlcy5zb3J0KAogICAgICAgICAgICAgICAga2V5PWxhbWJkYSBncm91cDogKAogICAgICAgICAgICAgICAgICAgIGFueSgibm9tdHAiIGluIG5hbWUubG93ZXIoKSBmb3IgbmFtZSBpbiBncm91cFswXSksCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMV0sCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMF1bMF0ubG93ZXIoKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgICAgICBmaWxlbmFtZXMsIHRvdGFsX3NpemUgPSBjYW5kaWRhdGVzWzBdCiAgICAgICAgICAgIHJldHVybiBNb2RlbFNlbGVjdGlvbigKICAgICAgICAgICAgICAgIHJlcG9faWQ9cmVwb19pZCwKICAgICAgICAgICAgICAgIHF1YW50PXF1YW50LAogICAgICAgICAgICAgICAgZmlsZXM9ZmlsZW5hbWVzLAogICAgICAgICAgICAgICAgcHJpbWFyeV9maWxlPWZpbGVuYW1lc1swXSwKICAgICAgICAgICAgICAgIHRvdGFsX3NpemVfYnl0ZXM9dG90YWxfc2l6ZSwKICAgICAgICAgICAgICAgIG1heF9tb2RlbF9naWI9bWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgKQogICAgcmFpc2UgTW9kZWxTZWxlY3Rpb25FcnJvcigKICAgICAgICAiTm8gY29tcGxldGUgUTRfS19NLCBJUTRfWFMsIG9yIFEzX0tfTSBHR1VGIGZpdHMgdGhlICIKICAgICAgICBmInttYXhfbW9kZWxfZ2liOmd9IEdpQiB3ZWlnaHQgYnVkZ2V0IGluIHtyZXBvX2lkfS4iCiAgICApCgoKZGVmIF9oYXNfZ2d1Zl9maWxlcyhhcGk6IEFueSwgcmVwb19pZDogc3RyLCB0b2tlbjogc3RyIHwgTm9uZSkgLT4gYm9vbDoKICAgICIiIkNoZWNrIGEgc2VhcmNoIHJlc3VsdCBmb3IgR0dVRiBmaWxlcyB3aXRob3V0IGRvd25sb2FkaW5nIG1vZGVsIGRhdGEuIiIiCiAgICB0cnk6CiAgICAgICAgaW5mbyA9IGFwaS5tb2RlbF9pbmZvKHJlcG9faWQsIGZpbGVzX21ldGFkYXRhPUZhbHNlLCB0b2tlbj10b2tlbikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gYW55KAogICAgICAgIHN0cihfZ2V0KGl0ZW0sICJyZmlsZW5hbWUiLCAiIikpLmxvd2VyKCkuZW5kc3dpdGgoIi5nZ3VmIikKICAgICAgICBmb3IgaXRlbSBpbiAoX2dldChpbmZvLCAic2libGluZ3MiLCBbXSkgb3IgW10pCiAgICApCgoKZGVmIGRpc2NvdmVyX2NhbmRpZGF0ZV9yZXBvcyhhcGk6IEFueSwgdG9rZW46IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W3N0cl06CiAgICAiIiJGaW5kIG90aGVyIHNpbWlsYXJseSBuYW1lZCB1bmNlbnNvcmVkIEdHVUYgcmVwb3NpdG9yaWVzIGZvciB1c2VyIGNob2ljZS4iIiIKICAgIHRyeToKICAgICAgICBtb2RlbHMgPSBhcGkubGlzdF9tb2RlbHMoc2VhcmNoPSJRd2VuMy44LTI3QiIsIGxpbWl0PTEwMCwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgVHlwZUVycm9yOgogICAgICAgIG1vZGVscyA9IGFwaS5saXN0X21vZGVscyhzZWFyY2g9IlF3ZW4zLjgtMjdCIikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIFtdCgogICAgY2FuZGlkYXRlczogbGlzdFtzdHJdID0gW10KICAgIGN1ZXMgPSAoInVuY2Vuc29yZWQiLCAidW5sZWFzaGVkIiwgImFibGl0ZXJhdCIsICJoZXJldGljIikKICAgIGZvciBtb2RlbCBpbiBtb2RlbHM6CiAgICAgICAgcmVwb19pZCA9IHN0cihfZ2V0KG1vZGVsLCAiaWQiLCAiIikgb3IgX2dldChtb2RlbCwgIm1vZGVsSWQiLCAiIikpCiAgICAgICAgbG93ZXJlZCA9IHJlcG9faWQubG93ZXIoKQogICAgICAgIGlmIG5vdCByZXBvX2lkIG9yIG5vdCBhbnkoY3VlIGluIGxvd2VyZWQgZm9yIGN1ZSBpbiBjdWVzKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBfaGFzX2dndWZfZmlsZXMoYXBpLCByZXBvX2lkLCB0b2tlbik6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKHJlcG9faWQpCiAgICAgICAgaWYgbGVuKGNhbmRpZGF0ZXMpID49IDEyOgogICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHNvcnRlZChzZXQoY2FuZGlkYXRlcyksIGtleT1zdHIubG93ZXIpCgoKZGVmIHJlc29sdmVfbW9kZWwoCiAgICBhcGk6IEFueSwKICAgIHJlcG9faWQ6IHN0ciwKICAgICosCiAgICB0b2tlbjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKICAgIG1pc3NpbmdfcmVwb19lcnJvcnM6IHR1cGxlW3R5cGVbQmFzZUV4Y2VwdGlvbl0sIC4uLl0gPSAoKSwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlZlcmlmeSBhIHJlcG9zaXRvcnksIG9yIHNob3cgc2VhcmNoIHJlc3VsdHMgaWYgaXQgaXMgbWlzc2luZy4iIiIKICAgIHRyeToKICAgICAgICBpbmZvID0gYXBpLm1vZGVsX2luZm8ocmVwb19pZCwgZmlsZXNfbWV0YWRhdGE9VHJ1ZSwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBpZiBub3QgbWlzc2luZ19yZXBvX2Vycm9ycyBvciBub3QgaXNpbnN0YW5jZShleGMsIG1pc3NpbmdfcmVwb19lcnJvcnMpOgogICAgICAgICAgICByYWlzZQogICAgICAgIGNhbmRpZGF0ZXMgPSBkaXNjb3Zlcl9jYW5kaWRhdGVfcmVwb3MoYXBpLCB0b2tlbikKICAgICAgICBzdWZmaXggPSAiXG4iLmpvaW4oZiIgIHtjYW5kaWRhdGV9IiBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXMpCiAgICAgICAgZ3VpZGFuY2UgPSAoCiAgICAgICAgICAgICJTZXQgTU9ERUxfUkVQTyBpbiB0aGUgS2FnZ2xlIG5vdGVib29rIHRvIG9uZSBvZiB0aGVzZSByZXBvc2l0b3JpZXMsICIKICAgICAgICAgICAgInRoZW4gcmVydW4gdGhlIHNlcnZlciBjZWxsLiIKICAgICAgICApCiAgICAgICAgaWYgc3VmZml4OgogICAgICAgICAgICBndWlkYW5jZSA9IGYie2d1aWRhbmNlfVxuQ2FuZGlkYXRlczpcbntzdWZmaXh9IgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoCiAgICAgICAgICAgIGYiUmVxdWVzdGVkIEh1Z2dpbmcgRmFjZSByZXBvc2l0b3J5IHdhcyBub3QgZm91bmQ6IHtyZXBvX2lkfS4ge2d1aWRhbmNlfSIKICAgICAgICApIGZyb20gZXhjCgogICAgc2libGluZ3MgPSBsaXN0KF9nZXQoaW5mbywgInNpYmxpbmdzIiwgW10pIG9yIFtdKQogICAgaWYgbm90IHNpYmxpbmdzOgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoZiJSZXBvc2l0b3J5IHtyZXBvX2lkfSBjb250YWlucyBubyBmaWxlcy4iKQogICAgcmV0dXJuIHNlbGVjdF9xdWFudChyZXBvX2lkLCBzaWJsaW5ncywgbWF4X21vZGVsX2dpYj1tYXhfbW9kZWxfZ2liKQoKCmRlZiBtYWluKGFyZ3Y6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lKSAtPiBpbnQ6CiAgICAiIiJSZXNvbHZlIGEgcmVwb3NpdG9yeSB0aHJvdWdoIHRoZSBIdWdnaW5nIEZhY2UgQVBJIGFuZCBwcmludCBpdHMgY2hvaWNlLiIiIgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwbyIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIk1PREVMX1JFUE8iLCBERUZBVUxUX1JFUE8pKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tYXgtbW9kZWwtZ2liIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1ERUZBVUxUX01BWF9NT0RFTF9HSUIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLWpzb24iLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJQcmludCBtYWNoaW5lLXJlYWRhYmxlIEpTT04uIgogICAgKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpCgogICAgdHJ5OgogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViLnV0aWxzIGltcG9ydCBSZXBvc2l0b3J5Tm90Rm91bmRFcnJvcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHByaW50KAogICAgICAgICAgICAiaHVnZ2luZ2ZhY2VfaHViIGlzIHJlcXVpcmVkIGluIHRoZSBLYWdnbGUgc2VydmVyIGVudmlyb25tZW50LiIsCiAgICAgICAgICAgIGZpbGU9c3lzLnN0ZGVyciwKICAgICAgICApCiAgICAgICAgcmV0dXJuIDIKCiAgICB0b2tlbiA9IG9zLmVudmlyb24uZ2V0KCJIRl9UT0tFTiIpIG9yIE5vbmUKICAgIHRyeToKICAgICAgICBzZWxlY3Rpb24gPSByZXNvbHZlX21vZGVsKAogICAgICAgICAgICBIZkFwaSgpLAogICAgICAgICAgICBhcmdzLnJlcG8sCiAgICAgICAgICAgIHRva2VuPXRva2VuLAogICAgICAgICAgICBtYXhfbW9kZWxfZ2liPWFyZ3MubWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgbWlzc2luZ19yZXBvX2Vycm9ycz0oUmVwb3NpdG9yeU5vdEZvdW5kRXJyb3IsKSwKICAgICAgICApCiAgICBleGNlcHQgTW9kZWxTZWxlY3Rpb25FcnJvciBhcyBleGM6CiAgICAgICAgcHJpbnQoc3RyKGV4YyksIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gMgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiSHVnZ2luZyBGYWNlIGxvb2t1cCBmYWlsZWQgKHt0eXBlKGV4YykuX19uYW1lX199KTsgY2hlY2sgS2FnZ2xlIEludGVybmV0ICIKICAgICAgICAgICAgImFjY2VzcyBhbmQgcmVwb3NpdG9yeSBwZXJtaXNzaW9ucy4iLAogICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgKQogICAgICAgIHJldHVybiAxCgogICAgcmVzdWx0ID0gYXNkaWN0KHNlbGVjdGlvbikKICAgIGlmIGFyZ3MuanNvbjoKICAgICAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCkpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYiUmVwb3NpdG9yeToge3NlbGVjdGlvbi5yZXBvX2lkfSIpCiAgICAgICAgcHJpbnQoZiJRdWFudDoge3NlbGVjdGlvbi5xdWFudH0iKQogICAgICAgIHByaW50KGYiVG90YWwgbW9kZWwgd2VpZ2h0IHNpemU6IHtzZWxlY3Rpb24udG90YWxfc2l6ZV9ieXRlcyAvIEdJQjouMmZ9IEdpQiIpCiAgICAgICAgcHJpbnQoZiJQcmltYXJ5IEdHVUY6IHtzZWxlY3Rpb24ucHJpbWFyeV9maWxlfSIpCiAgICAgICAgcHJpbnQoIkZpbGVzIHRvIGRvd25sb2FkOiIpCiAgICAgICAgZm9yIGZpbGVuYW1lIGluIHNlbGVjdGlvbi5maWxlczoKICAgICAgICAgICAgcHJpbnQoZiIgIHtmaWxlbmFtZX0iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg=="
}
for filename, content in embedded_files.items():
    file_path = server_dir / filename
    file_path.write_bytes(base64.b64decode(content))
    if filename.endswith('.sh'):
        file_path.chmod(0o700)
print(f"Prepared server scripts in {server_dir}")


In [ ]:
%cd /kaggle/working/qwenfree_server
!bash bootstrap.sh
